# LinkedIn Hiring Rate

In [1]:
from pathlib import Path

import altair as alt
import attaviz
import pandas as pd

attaviz.enable()
alt.data_transformers.enable("vegafusion")


def find_project_root(marker="pyproject.toml"):
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "LinkedIn"
PROCESSED_PATH = DATA_PATH / "processed"
PROCESSED_PATH.mkdir(exist_ok=True)

LHR_FILE = (
    DATA_PATH / "LinkedIn Hiring Rate" / "LinkedIn_LHR by Industry SA_Aug2026.xlsx"
)

LHR_VALUE = "LHR (SA)"
BASELINE = 1.0

WEST_AFRICA = ["Ghana", "Nigeria"]
COMPARATORS = ["India", "Kenya", "South Africa"]
COUNTRIES = WEST_AFRICA + COMPARATORS

SOURCE_NOTE = (
    "Source: LinkedIn Economic Graph, seasonally adjusted hiring rate "
    "(release Aug 2026).\n"
    "The index is normalised so that the 2016 monthly average equals 1.0."
)

In [2]:
def tidy_lhr(excel_file, sheet_name, names, countries=None):
    """Read one LHR sheet and return a tidy frame."""
    return (
        pd.read_excel(excel_file, sheet_name=sheet_name, header=3)
        .drop(columns=["Unnamed: 0"])
        .set_axis(names, axis="columns")
        .loc[lambda d: d["Country"].ne("Country")]
        .dropna(subset=names)
        .assign(
            Month=lambda d: pd.to_datetime(d["Month"]),
            Country=lambda d: d["Country"].str.strip(),
            **{LHR_VALUE: lambda d: pd.to_numeric(d[LHR_VALUE])},
        )
        .loc[lambda d: d["Country"].isin(countries) if countries else slice(None)]
        .sort_values(names[:-1])
        .reset_index(drop=True)
    )


def baseline_rule(data, value=BASELINE):
    """A single dashed rule at the 2016 baseline, drawn once per panel."""
    return (
        alt.Chart(data)
        .transform_aggregate(_rows="count()")
        .mark_rule(color=attaviz.REFERENCE, strokeDash=[4, 4])
        .encode(y=alt.datum(value))
    )

In [3]:
country_lhr = tidy_lhr(
    LHR_FILE,
    sheet_name="2A - LHR SA by Ctry",
    names=["Month", "Country", LHR_VALUE],
    countries=COUNTRIES,
)
country_lhr.to_csv(PROCESSED_PATH / "lhr_country.csv", index=False)

print(country_lhr.shape)
country_lhr.head()

(575, 3)


,Month,Country,LHR (SA)
0,2017-01-01,Ghana,0.9971
1,2017-01-01,India,1.0066
2,2017-01-01,Kenya,0.7948
3,2017-01-01,Nigeria,0.9623
4,2017-01-01,South Africa,0.9742


In [4]:
country_lhr.groupby("Country").agg(
    months=("Month", "size"),
    first=("Month", "min"),
    last=("Month", "max"),
    mean_index=(LHR_VALUE, "mean"),
).assign(mean_index=lambda d: d["mean_index"].round(2))

,months,first,last,mean_index
Country,,,,
Ghana,115,2017-01-01,2026-07-01,1.36
India,115,2017-01-01,2026-07-01,1.78
Kenya,115,2017-01-01,2026-07-01,1.02
Nigeria,115,2017-01-01,2026-07-01,1.97
South Africa,115,2017-01-01,2026-07-01,0.99


## LinkedIn Hiring Rate by country

In [5]:
countries = sorted(country_lhr["Country"].unique())
zoom = alt.selection_interval(bind="scales", encodings=["x"])

line = (
    alt.Chart(country_lhr)
    .mark_line()
    .encode(
        x=alt.X("Month:T", title=None),
        y=alt.Y(f"{LHR_VALUE}:Q", title="Hiring rate index (2016 = 1.0)"),
        color=alt.Color("Country:N", legend=None),
        tooltip=[
            alt.Tooltip("Country:N"),
            alt.Tooltip("Month:T", format="%b %Y"),
            alt.Tooltip(f"{LHR_VALUE}:Q", format=".2f"),
        ],
    )
    .add_params(zoom)
)

chart = (
    (baseline_rule(country_lhr) + line)
    .properties(width=240, height=170)
    .facet(
        facet=alt.Facet("Country:N", title=None, sort=countries),
        columns=3,
        title="LinkedIn Hiring Rate, seasonally adjusted index",
    )
)

attaviz.add_caption(chart, SOURCE_NOTE)

alt.VConcatChart(...)